# Instalasi dan Verifikasi Dependensi

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import time

# Cek versi PyTorch
print(f"PyTorch Version : {torch.__version__}")

# Cek ketersediaan GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device          : {device}")

In [ ]:
# ============================
# MODUL PEMBUNGKUS RNN UNTUK nn.Sequential()
# ============================

# Pembungkus RNN/GRU: mengekstrak output sequence dari tuple (output, h_n)
# Ingat: nn.RNN dan nn.GRU mengembalikan (output, hidden_state)

class ExtractRNNOutput(nn.Module):
    """Mengekstrak output sequence dari layer RNN/GRU.
    Output shape: (batch, seq_len, hidden_size) jika batch_first=True
    """
    __init__ = lambda self: super().__init__()
    forward = lambda self, x: x[0]  # Ambil elemen pertama dari tuple (output)

# Pembungkus LSTM: mengekstrak output sequence dari tuple (output, (h_n, c_n))
# Ingat: nn.LSTM mengembalikan (output, (hidden_state, cell_state))

class ExtractLSTMOutput(nn.Module):
    """Mengekstrak output sequence dari layer LSTM.
    Output shape: (batch, seq_len, hidden_size) jika batch_first=True
    """
    __init__ = lambda self: super().__init__()
    forward = lambda self, x: x[0]  # Ambil output (sama seperti RNN/GRU)

# Mengambil output di time step TERAKHIR saja (untuk klasifikasi many-to-one)
# Dari shape (batch, seq_len, hidden_size) → (batch, hidden_size)

class SelectLastTimeStep(nn.Module):
    """Mengambil output pada time step terakhir dari sequence.
    Input:  (batch, seq_len, hidden_size)
    Output: (batch, hidden_size)
    """
    __init__ = lambda self: super().__init__()
    forward = lambda self, x: x[:, -1, :]  # Ambil time step terakhir

print("Modul pembungkus berhasil didefinisikan!")
print("   - ExtractRNNOutput  : Mengekstrak output dari RNN/GRU")
print("   - ExtractLSTMOutput : Mengekstrak output dari LSTM")
print("   - SelectLastTimeStep: Mengambil output time step terakhir")

In [ ]:
# ============================
# DEMONSTRASI DIMENSI DATA RNN
# ============================

# Data RNN memiliki 3 dimensi: (batch_size, sequence_length, input_size)
#
#   batch_size     = berapa banyak sampel per batch
#   sequence_length = berapa banyak langkah waktu (time steps)
#   input_size     = berapa banyak fitur per langkah waktu

# Contoh: 4 kalimat, masing-masing 10 kata, setiap kata direpresentasikan 32 fitur
batch_size = 4
seq_len = 10
input_size = 32

dummy_input = torch.randn(batch_size, seq_len, input_size)
print(f"Input shape: {dummy_input.shape}")
# Output: torch.Size([4, 10, 32])
# Artinya: 4 sampel, 10 time steps, 32 features per time step

# Membuat layer RNN sederhana
rnn_layer = nn.RNN(input_size=32, hidden_size=64, batch_first=True)

# Forward pass
output, h_n = rnn_layer(dummy_input)

print(f"\nOutput shape : {output.shape}")   # (4, 10, 64) — output di setiap time step
print(f"h_n shape    : {h_n.shape}")        # (1, 4, 64)  — hidden state terakhir

# output[:, -1, :] == h_n[0] → output di time step terakhir = hidden state terakhir
print(f"\nApakah output[:,-1,:] == h_n[0]?")
print(f"  {torch.allclose(output[:, -1, :], h_n[0], atol=1e-6)}")  # True!

In [ ]:
# ============================
# DEMONSTRASI RNN CELL (LANGKAH DEMI LANGKAH)
# ============================

torch.manual_seed(42)

# RNNCell memproses SATU time step pada satu waktu
rnn_cell = nn.RNNCell(input_size=3, hidden_size=5)

# Input sequence: 4 time steps, masing-masing 3 fitur
# (Bayangkan: 4 kata, masing-masing diwakili 3 angka)
x_sequence = torch.tensor([
    [1.0, 0.0, 0.0],   # Time step 0
    [0.0, 1.0, 0.0],   # Time step 1
    [0.0, 0.0, 1.0],   # Time step 2
    [1.0, 1.0, 0.0],   # Time step 3
])

print("Input Sequence (4 time steps, 3 features):")
print(x_sequence)
print(f"Shape: {x_sequence.shape}")

# Proses langkah demi langkah
h = torch.zeros(1, 5)  # Hidden state awal (1 sampel, 5 hidden units)
print(f"\nHidden state awal: {h.shape}")
print(f"  h₀ = {h.squeeze().detach().numpy()}")

print("\n--- Proses per Time Step ---")
for t in range(len(x_sequence)):
    x_t = x_sequence[t].unsqueeze(0)  # (1, 3) — tambah dimensi batch
    h = rnn_cell(x_t, h)               # h_new = tanh(W_xh * x_t + W_hh * h_old + b)

    print(f"  t={t} | Input: {x_t.squeeze().numpy()} → "
          f"Hidden: [{', '.join([f'{v:.3f}' for v in h.squeeze().detach().numpy()])}]")

print(f"\nHidden state akhir mengandung 'ringkasan' dari seluruh sequence!")

In [ ]:
# ============================
# PERBANDINGAN RNN vs LSTM vs GRU
# ============================

torch.manual_seed(42)

# Input: 2 sampel, 6 time steps, 4 fitur
batch_input = torch.randn(2, 6, 4)
print(f"Input shape: {batch_input.shape}")

# Buat 3 model
rnn = nn.RNN(input_size=4, hidden_size=8, batch_first=True)
lstm = nn.LSTM(input_size=4, hidden_size=8, batch_first=True)
gru = nn.GRU(input_size=4, hidden_size=8, batch_first=True)

# Forward pass
rnn_out, rnn_hn = rnn(batch_input)
lstm_out, (lstm_hn, lstm_cn) = lstm(batch_input)
gru_out, gru_hn = gru(batch_input)

print("\n=== OUTPUT SHAPES ===")
print(f"{'Model':<8} | {'Output':<25} | {'Hidden State':<20} | {'Cell State':<20}")
print("-" * 80)
print(f"{'RNN':<8} | {str(rnn_out.shape):<25} | {str(rnn_hn.shape):<20} | {'N/A':<20}")
print(f"{'LSTM':<8} | {str(lstm_out.shape):<25} | {str(lstm_hn.shape):<20} | {str(lstm_cn.shape):<20}")
print(f"{'GRU':<8} | {str(gru_out.shape):<25} | {str(gru_hn.shape):<20} | {'N/A':<20}")

print("\n=== JUMLAH PARAMETER ===")
rnn_params = sum(p.numel() for p in rnn.parameters())
lstm_params = sum(p.numel() for p in lstm.parameters())
gru_params = sum(p.numel() for p in gru.parameters())
print(f"RNN  : {rnn_params:,} parameter")
print(f"LSTM : {lstm_params:,} parameter (~{lstm_params/rnn_params:.1f}× RNN)")
print(f"GRU  : {gru_params:,} parameter (~{gru_params/rnn_params:.1f}× RNN)")

In [ ]:
# ============================
# MEMBUAT SLIDING WINDOW DATASET
# ============================

# Fungsi sliding window diimplementasikan dengan list comprehension (tanpa def)
# Ini adalah teknik fundamental untuk mempersiapkan data time series untuk RNN

# Contoh data: sinyal sederhana
raw_data = torch.sin(torch.linspace(0, 4 * np.pi, 100))

# Sliding window: ubah [a, b, c, d, e, f, ...] menjadi pasangan input-target
# Jika window_size=5:
#   Input:  [a, b, c, d, e]  → Target: f
#   Input:  [b, c, d, e, f]  → Target: g
#   ...

window_size = 10

# Membuat pasangan (input_window, target) tanpa fungsi def
X_windows = torch.stack([raw_data[i:i+window_size] for i in range(len(raw_data) - window_size)])
y_targets = raw_data[window_size:]

# Tambahkan dimensi fitur: (samples, seq_len) → (samples, seq_len, 1)
X_windows = X_windows.unsqueeze(-1)  # (90, 10, 1)
y_targets = y_targets.unsqueeze(-1)  # (90, 1)

print(f"Data asli     : {raw_data.shape}")
print(f"X (windows)   : {X_windows.shape}")  # (90, 10, 1)
print(f"y (targets)   : {y_targets.shape}")   # (90, 1)

# Visualisasi sliding window
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot sinyal asli
axes[0].plot(raw_data.numpy(), color='steelblue', linewidth=2)
axes[0].set_title('Sinyal Asli', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Langkah Waktu', fontsize=12)
axes[0].set_ylabel('Nilai', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Plot beberapa window
colors = ['crimson', 'green', 'orange', 'purple']
for i, idx in enumerate([0, 20, 40, 60]):
    window = X_windows[idx].squeeze().numpy()
    target = y_targets[idx].item()
    axes[1].plot(range(idx, idx + window_size), window,
                 color=colors[i], linewidth=2, marker='o', markersize=4,
                 label=f'Window {idx} → target={target:.2f}')
    axes[1].plot(idx + window_size, target, 'x', color=colors[i],
                 markersize=12, markeredgewidth=3)

axes[1].set_title('Contoh Sliding Window (size=10)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Langkah Waktu', fontsize=12)
axes[1].set_ylabel('Nilai', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DATASET: SINYAL SINUSOIDAL
# ============================

torch.manual_seed(42)
np.random.seed(42)

# Membuat sinyal sinusoidal dengan noise
n_points = 1000
t = np.linspace(0, 20 * np.pi, n_points)
signal = np.sin(t) + 0.1 * np.random.randn(n_points)
signal = torch.FloatTensor(signal)

print(f"Jumlah data point: {len(signal)}")

# Normalisasi ke range [-1, 1]
signal = (signal - signal.mean()) / signal.std()

# Visualisasi
plt.figure(figsize=(14, 4))
plt.plot(signal.numpy(), color='steelblue', linewidth=0.8, alpha=0.8)
plt.title('Sinyal Sinusoidal + Noise', fontsize=14, fontweight='bold')
plt.xlabel('Langkah Waktu', fontsize=12)
plt.ylabel('Nilai (Normalized)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# SLIDING WINDOW + SPLIT DATA
# ============================

seq_length = 30  # Gunakan 30 langkah terakhir untuk prediksi

# Membuat pasangan (window, target)
X = torch.stack([signal[i:i+seq_length] for i in range(len(signal) - seq_length)])
y = signal[seq_length:]

# Tambah dimensi fitur: (samples, seq_len) → (samples, seq_len, 1)
X = X.unsqueeze(-1)  # (970, 30, 1)
y = y.unsqueeze(-1)  # (970, 1)

# Split: 80% train, 20% test (secara berurutan, BUKAN acak — penting untuk time series!)
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}  |  y_test : {y_test.shape}")

# DataLoader
batch_size = 32
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# ============================
# MODEL: RNN SEDERHANA
# ============================

# Arsitektur:
#   Input (batch, 30, 1) — 30 time steps, 1 fitur
#     ↓
#   RNN(input=1, hidden=64, batch_first=True)
#     ↓
#   ExtractRNNOutput → (batch, 30, 64)
#     ↓
#   SelectLastTimeStep → (batch, 64)  — ambil output terakhir
#     ↓
#   Linear(64, 1) — prediksi 1 nilai

model_rnn = nn.Sequential(
    nn.RNN(input_size=1, hidden_size=64, num_layers=1, batch_first=True),
    ExtractRNNOutput(),          # (batch,30,64) — ambil output, buang hidden state
    SelectLastTimeStep(),        # (batch,64)    — ambil time step terakhir
    nn.Linear(64, 1)             # (batch,1)     — prediksi 1 nilai
)

print("Arsitektur Model RNN:")
print(model_rnn)

total_params = sum(p.numel() for p in model_rnn.parameters())
print(f"\nTotal parameter: {total_params:,}")

In [ ]:
# ============================
# VERIFIKASI DIMENSI
# ============================

dummy = torch.randn(4, 30, 1)  # 4 sampel, 30 time steps, 1 fitur
print("--- Verifikasi Dimensi per Layer ---")
print(f"Input                : {dummy.shape}")

# Layer 0: nn.RNN
rnn_out = model_rnn[0](dummy)
print(f"nn.RNN output        : {rnn_out[0].shape} (output), {rnn_out[1].shape} (h_n)")

# Layer 1: ExtractRNNOutput
extracted = model_rnn[1](rnn_out)
print(f"ExtractRNNOutput     : {extracted.shape}")

# Layer 2: SelectLastTimeStep
last_step = model_rnn[2](extracted)
print(f"SelectLastTimeStep   : {last_step.shape}")

# Layer 3: nn.Linear
final_out = model_rnn[3](last_step)
print(f"nn.Linear(64, 1)     : {final_out.shape}")

# Verifikasi end-to-end
with torch.no_grad():
    full_out = model_rnn(dummy)
print(f"\nEnd-to-end output    : {full_out.shape}")  # (4, 1) ✓

In [ ]:
# ============================
# LOSS FUNCTION & OPTIMIZER
# ============================

criterion = nn.MSELoss()
optimizer = optim.Adam(model_rnn.parameters(), lr=0.001)

# ============================
# TRAINING LOOP
# ============================

num_epochs = 100
train_losses = []
test_losses = []

print("Mulai Training RNN...")
print("=" * 65)

for epoch in range(num_epochs):
    # ---- TRAINING PHASE ----
    model_rnn.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Forward pass
        y_pred = model_rnn(X_batch)
        loss = criterion(y_pred, y_batch)

        # Backward pass & update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ---- TESTING PHASE ----
    model_rnn.eval()
    test_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            y_pred = model_rnn(X_batch)
            loss = criterion(y_pred, y_batch)
            test_loss += loss.item()

    test_loss = test_loss / len(test_loader)
    test_losses.append(test_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"Train Loss: {train_loss:.6f} | "
              f"Test Loss: {test_loss:.6f}")

print("=" * 65)
print("Training Selesai!")

In [ ]:
# ============================
# PLOT 1: TRAINING LOSS
# ============================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(train_losses, label='Train Loss', color='crimson', linewidth=2)
axes[0].plot(test_losses, label='Test Loss', color='steelblue',
             linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Training vs Test Loss — RNN', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# ============================
# PLOT 2: PREDIKSI vs AKTUAL
# ============================

model_rnn.eval()
with torch.no_grad():
    y_pred_train = model_rnn(X_train).squeeze().numpy()
    y_pred_test = model_rnn(X_test).squeeze().numpy()

y_actual_train = y_train.squeeze().numpy()
y_actual_test = y_test.squeeze().numpy()

# Gabungkan index untuk plotting
train_idx = range(seq_length, seq_length + len(y_actual_train))
test_idx = range(seq_length + len(y_actual_train),
                 seq_length + len(y_actual_train) + len(y_actual_test))

axes[1].plot(train_idx, y_actual_train, color='steelblue',
             linewidth=1, alpha=0.5, label='Aktual (Train)')
axes[1].plot(test_idx, y_actual_test, color='steelblue',
             linewidth=2, label='Aktual (Test)')
axes[1].plot(test_idx, y_pred_test, color='crimson',
             linewidth=2, linestyle='--', label='Prediksi (Test)')
axes[1].axvline(x=seq_length + len(y_actual_train), color='gray',
                linestyle=':', linewidth=2, label='Train/Test Split')
axes[1].set_xlabel('Langkah Waktu', fontsize=12)
axes[1].set_ylabel('Nilai', fontsize=12)
axes[1].set_title('Prediksi RNN — Sinyal Sinusoidal', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Hitung metrik
mse = np.mean((y_actual_test - y_pred_test) ** 2)
mae = np.mean(np.abs(y_actual_test - y_pred_test))
print(f"\nMetrik pada Data Test:")
print(f"   MSE (Mean Squared Error) : {mse:.6f}")
print(f"   MAE (Mean Absolute Error): {mae:.6f}")

In [ ]:
# ============================
# DATASET: SINYAL MULTI-FREKUENSI
# ============================

torch.manual_seed(42)
np.random.seed(42)

# Sinyal gabungan: kombinasi 2 frekuensi + tren + noise
n_points = 1500
t = np.linspace(0, 30 * np.pi, n_points)

# Komponen sinyal:
# - Frekuensi rendah (pola jangka panjang)
# - Frekuensi tinggi (pola jangka pendek)
# - Noise acak
signal_complex = (
    0.6 * np.sin(t) +           # Frekuensi rendah
    0.3 * np.sin(3 * t) +       # Frekuensi tinggi
    0.05 * np.random.randn(n_points)  # Noise
)

signal_complex = torch.FloatTensor(signal_complex)

# Normalisasi
signal_complex = (signal_complex - signal_complex.mean()) / signal_complex.std()

# Visualisasi
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(signal_complex.numpy(), color='steelblue', linewidth=0.8)
axes[0].set_title('Sinyal Multi-Frekuensi (Gabungan)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Langkah Waktu', fontsize=12)
axes[0].set_ylabel('Nilai', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Zoom-in 200 data point pertama
axes[1].plot(signal_complex[:200].numpy(), color='steelblue', linewidth=1.5, marker='o',
             markersize=2)
axes[1].set_title('Zoom-in: 200 Data Point Pertama', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Langkah Waktu', fontsize=12)
axes[1].set_ylabel('Nilai', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Total data points: {len(signal_complex)}")

In [ ]:
# ============================
# SLIDING WINDOW + SPLIT
# ============================

seq_length = 50  # Window lebih panjang untuk menangkap pola multi-frekuensi

# Membuat windows
X = torch.stack([signal_complex[i:i+seq_length] for i in range(len(signal_complex) - seq_length)])
y = signal_complex[seq_length:]

X = X.unsqueeze(-1)  # (samples, 50, 1)
y = y.unsqueeze(-1)  # (samples, 1)

# Split: 80% train, 20% test
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

print(f"X_train: {X_train.shape}  |  y_train: {y_train.shape}")
print(f"X_test : {X_test.shape}  |  y_test : {y_test.shape}")

# DataLoader
batch_size = 64
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
# ============================
# MODEL: LSTM
# ============================

# Arsitektur:
#   Input (batch, 50, 1) — 50 time steps, 1 fitur
#     ↓
#   LSTM(input=1, hidden=128, num_layers=2, dropout=0.2)
#     ↓
#   ExtractLSTMOutput → (batch, 50, 128)
#     ↓
#   SelectLastTimeStep → (batch, 128)
#     ↓
#   Linear(128, 64) + ReLU
#     ↓
#   Linear(64, 1)

model_lstm = nn.Sequential(
    nn.LSTM(input_size=1, hidden_size=128, num_layers=2,
            batch_first=True, dropout=0.2),
    ExtractLSTMOutput(),         # (batch, 50, 128)
    SelectLastTimeStep(),        # (batch, 128)
    nn.Linear(128, 64),          # Hidden layer tambahan
    nn.ReLU(),
    nn.Linear(64, 1)             # Output
)

print("Arsitektur Model LSTM:")
print(model_lstm)

total_params = sum(p.numel() for p in model_lstm.parameters())
print(f"\nTotal parameter: {total_params:,}")

In [ ]:
# ============================
# LOSS & OPTIMIZER
# ============================

criterion = nn.MSELoss()
optimizer = optim.Adam(model_lstm.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)

# ============================
# TRAINING LOOP
# ============================

num_epochs = 100
train_losses = []
test_losses = []

print("Mulai Training LSTM...")
print("=" * 65)

for epoch in range(num_epochs):
    # ---- TRAINING PHASE ----
    model_lstm.train()
    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        y_pred = model_lstm(X_batch)
        loss = criterion(y_pred, y_batch)

        optimizer.zero_grad()
        loss.backward()

        # Gradient Clipping: mencegah exploding gradient pada RNN!
        torch.nn.utils.clip_grad_norm_(model_lstm.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)

    # ---- TESTING PHASE ----
    model_lstm.eval()
    test_loss = 0.0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            y_pred = model_lstm(X_batch)
            loss = criterion(y_pred, y_batch)
            test_loss += loss.item()

    test_loss = test_loss / len(test_loader)
    test_losses.append(test_loss)

    scheduler.step()

    if (epoch + 1) % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch [{epoch+1:3d}/{num_epochs}] | "
              f"LR: {current_lr:.6f} | "
              f"Train Loss: {train_loss:.6f} | "
              f"Test Loss: {test_loss:.6f}")

print("=" * 65)
print("Training Selesai!")

In [ ]:
# ============================
# PLOT TRAINING LOSS & PREDIKSI
# ============================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(train_losses, label='Train Loss', color='crimson', linewidth=2)
axes[0].plot(test_losses, label='Test Loss', color='steelblue',
             linewidth=2, linestyle='--')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss (MSE)', fontsize=12)
axes[0].set_title('Training vs Test Loss — LSTM', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Prediksi vs Aktual
model_lstm.eval()
with torch.no_grad():
    y_pred_test = model_lstm(X_test).squeeze().numpy()

y_actual_test = y_test.squeeze().numpy()

axes[1].plot(y_actual_test, label='Aktual', color='steelblue', linewidth=2)
axes[1].plot(y_pred_test, label='Prediksi LSTM', color='crimson',
             linewidth=2, linestyle='--')
axes[1].set_xlabel('Langkah Waktu (Test Set)', fontsize=12)
axes[1].set_ylabel('Nilai', fontsize=12)
axes[1].set_title('Prediksi LSTM — Sinyal Multi-Frekuensi', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Metrik
mse = np.mean((y_actual_test - y_pred_test) ** 2)
mae = np.mean(np.abs(y_actual_test - y_pred_test))
print(f"\nMetrik LSTM pada Data Test:")
print(f"   MSE : {mse:.6f}")
print(f"   MAE : {mae:.6f}")

In [ ]:
# ============================
# PREDIKSI MULTI-STEP KE DEPAN
# ============================

# Prediksi di mana kita menggunakan output sebelumnya sebagai input untuk langkah berikutnya
# Ini menunjukkan kemampuan LSTM memprediksi beberapa langkah ke depan

model_lstm.eval()
n_future = 100  # Prediksi 100 langkah ke depan

# Mulai dari window terakhir dari data test
last_window = X_test[-1].clone()  # (50, 1)
future_preds = []

with torch.no_grad():
    current_window = last_window.unsqueeze(0)  # (1, 50, 1)

    for step in range(n_future):
        # Prediksi 1 langkah ke depan
        next_val = model_lstm(current_window)  # (1, 1)
        future_preds.append(next_val.item())

        # Geser window: buang time step pertama, tambahkan prediksi di akhir
        new_step = next_val.unsqueeze(0)  # (1, 1, 1)
        current_window = torch.cat([current_window[:, 1:, :], new_step], dim=1)

future_preds = np.array(future_preds)

# Plot
plt.figure(figsize=(14, 5))
n_context = 100  # Tampilkan 100 data terakhir sebagai konteks

plt.plot(range(n_context), y_actual_test[-n_context:],
         color='steelblue', linewidth=2, label='Data Aktual')
plt.plot(range(n_context, n_context + n_future), future_preds,
         color='crimson', linewidth=2, linestyle='--', label='Prediksi Masa Depan')
plt.axvline(x=n_context, color='gray', linestyle=':', linewidth=2, alpha=0.7)
plt.fill_between(range(n_context, n_context + n_future),
                 future_preds - 0.3, future_preds + 0.3,
                 alpha=0.1, color='crimson', label='Interval Ketidakpastian')
plt.xlabel('Langkah Waktu', fontsize=12)
plt.ylabel('Nilai', fontsize=12)
plt.title('Prediksi Multi-Step ke Depan — LSTM', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# DATASET MNIST UNTUK GRU
# ============================

torch.manual_seed(42)

# MNIST: gambar 28×28 → diperlakukan sebagai sekuens 28 langkah, 28 fitur
# TIDAK perlu Flatten!

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True,
                                download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False,
                               download=True, transform=transform)

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Cek dimensi
sample_img, sample_label = train_dataset[0]
print(f"Image shape       : {sample_img.shape}")       # (1, 28, 28)
print(f"Sebagai sekuens   : ({sample_img.shape[1]}, {sample_img.shape[2]})")
print(f"  = {sample_img.shape[1]} time steps, {sample_img.shape[2]} features per step")
print(f"Jumlah kelas      : 10 (digit 0-9)")
print(f"Training samples  : {len(train_dataset)}")
print(f"Test samples      : {len(test_dataset)}")

In [ ]:
# ============================
# VISUALISASI MNIST SEBAGAI SEKUENS
# ============================

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('MNIST: Gambar sebagai Sekuens 28 Baris',
             fontsize=16, fontweight='bold', y=1.02)

for i, ax in enumerate(axes.flat):
    img, label = train_dataset[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f'Digit: {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Fitur (28 piksel)', fontsize=9)
    ax.set_ylabel('Time Step (28 baris)', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# MODUL RESHAPE: (batch, 1, 28, 28) → (batch, 28, 28)
# ============================

# Gambar MNIST berbentuk (batch, 1, 28, 28) — perlu diubah ke (batch, 28, 28)
# agar bisa diproses RNN sebagai 28 time steps × 28 features

class ReshapeForRNN(nn.Module):
    """Mengubah gambar (batch, C, H, W) menjadi sekuens (batch, H, W)
    untuk diproses oleh RNN/LSTM/GRU.
    Gambar dibaca baris per baris: H = time steps, W = features.
    """
    __init__ = lambda self: super().__init__()
    forward = lambda self, x: x.squeeze(1)  # Buang channel dimension: (B,1,28,28)→(B,28,28)

print("ReshapeForRNN berhasil didefinisikan!")
print("   (batch, 1, 28, 28) → (batch, 28, 28)")

In [ ]:
# ============================
# MODEL: GRU UNTUK KLASIFIKASI MNIST
# ============================

# Arsitektur:
#   Input (batch, 1, 28, 28) — gambar MNIST
#     ↓
#   ReshapeForRNN → (batch, 28, 28) — 28 time steps, 28 fitur
#     ↓
#   GRU(input=28, hidden=128, num_layers=2, dropout=0.3)
#     ↓
#   ExtractRNNOutput → (batch, 28, 128)
#     ↓
#   SelectLastTimeStep → (batch, 128)
#     ↓
#   Linear(128, 64) + ReLU + Dropout(0.3)
#     ↓
#   Linear(64, 10) — 10 kelas digit

model_gru = nn.Sequential(
    # Reshape gambar menjadi sekuens
    ReshapeForRNN(),                 # (batch,1,28,28) → (batch,28,28)

    # GRU layer
    nn.GRU(input_size=28, hidden_size=128, num_layers=2,
           batch_first=True, dropout=0.3),
    ExtractRNNOutput(),              # (batch,28,128)
    SelectLastTimeStep(),            # (batch,128)

    # Classifier
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 10)               # 10 kelas (TANPA Softmax!)
)

# Pindahkan ke device
model_gru = model_gru.to(device)

print("Arsitektur Model GRU MNIST:")
print(model_gru)

total_params = sum(p.numel() for p in model_gru.parameters())
print(f"\nTotal parameter: {total_params:,}")

In [ ]:
# ============================
# VERIFIKASI DIMENSI
# ============================

dummy = torch.randn(4, 1, 28, 28).to(device)  # 4 gambar MNIST
print("--- Verifikasi Dimensi per Layer ---")
print(f"Input                : {dummy.shape}")

x = model_gru[0](dummy)   # ReshapeForRNN
print(f"ReshapeForRNN        : {x.shape}")

x_tuple = model_gru[1](x)  # nn.GRU
print(f"nn.GRU output        : {x_tuple[0].shape} (output), {x_tuple[1].shape} (h_n)")

x = model_gru[2](x_tuple)  # ExtractRNNOutput
print(f"ExtractRNNOutput     : {x.shape}")

x = model_gru[3](x)        # SelectLastTimeStep
print(f"SelectLastTimeStep   : {x.shape}")

x = model_gru[4](x)        # Linear(128,64)
print(f"Linear(128→64)       : {x.shape}")

x = model_gru[5](x)        # ReLU
print(f"ReLU                 : {x.shape}")

x = model_gru[6](x)        # Dropout
print(f"Dropout(0.3)         : {x.shape}")

x = model_gru[7](x)        # Linear(64,10)
print(f"Linear(64→10)        : {x.shape}")

# End-to-end
with torch.no_grad():
    out = model_gru(dummy)
print(f"\nEnd-to-end output    : {out.shape}")  # (4, 10) ✓

In [ ]:
# ============================
# LOSS, OPTIMIZER, SCHEDULER
# ============================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_gru.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# ============================
# TRAINING LOOP
# ============================

num_epochs = 15
train_losses = []
test_losses = []
train_accs = []
test_accs = []

print("Mulai Training GRU MNIST...")
print("=" * 80)

for epoch in range(num_epochs):
    # ---- TRAINING PHASE ----
    model_gru.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model_gru(images)
        loss = criterion(outputs, labels)

        # Backward pass & update
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_gru.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total * 100
    train_losses.append(train_loss)
    train_accs.append(train_acc)

    # ---- TESTING PHASE ----
    model_gru.eval()
    test_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_gru(images)
            loss = criterion(outputs, labels)

            test_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_loss = test_loss / len(test_loader)
    test_acc = correct / total * 100
    test_losses.append(test_loss)
    test_accs.append(test_acc)

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
          f"LR: {current_lr:.6f} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2f}%")

print("=" * 80)
print(f"Akurasi Akhir → Train: {train_accs[-1]:.2f}% | Test: {test_accs[-1]:.2f}%")

In [ ]:
# ============================
# PLOT LOSS & ACCURACY
# ============================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(range(1, num_epochs+1), train_losses, 'o-',
             color='crimson', linewidth=2, label='Train Loss')
axes[0].plot(range(1, num_epochs+1), test_losses, 's--',
             color='steelblue', linewidth=2, label='Test Loss')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training vs Test Loss — GRU MNIST', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(range(1, num_epochs+1), train_accs, 'o-',
             color='crimson', linewidth=2, label='Train Accuracy')
axes[1].plot(range(1, num_epochs+1), test_accs, 's--',
             color='steelblue', linewidth=2, label='Test Accuracy')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Train vs Test Accuracy — GRU MNIST', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# EVALUASI DETAIL
# ============================

model_gru.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_gru(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Classification Report
print("\nClassification Report — GRU MNIST:")
print("=" * 60)
print(classification_report(all_labels, all_preds,
                             target_names=[f'Digit {i}' for i in range(10)]))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[str(i) for i in range(10)],
            yticklabels=[str(i) for i in range(10)])
plt.xlabel('Prediksi', fontsize=13)
plt.ylabel('Aktual', fontsize=13)
plt.title('Confusion Matrix — GRU MNIST', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# VISUALISASI PREDIKSI
# ============================

model_gru.eval()
test_images, test_labels = next(iter(test_loader))
test_images_device = test_images.to(device)

with torch.no_grad():
    outputs = model_gru(test_images_device)
    probabilities = torch.softmax(outputs, dim=1)
    _, predictions = torch.max(outputs, 1)

fig, axes = plt.subplots(2, 5, figsize=(16, 7))
fig.suptitle('Prediksi GRU pada Data Test MNIST',
             fontsize=16, fontweight='bold', y=1.02)

for i, ax in enumerate(axes.flat):
    img = test_images[i].squeeze().cpu().numpy()
    true_label = test_labels[i].item()
    pred_label = predictions[i].cpu().item()
    confidence = probabilities[i][pred_label].cpu().item() * 100

    ax.imshow(img, cmap='gray')
    color = 'green' if pred_label == true_label else 'red'
    ax.set_title(f'Pred: {pred_label} ({confidence:.1f}%)\nTrue: {true_label}',
                 fontsize=11, fontweight='bold', color=color)
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ============================
# TIGA MODEL UNTUK PERBANDINGAN
# ============================

torch.manual_seed(42)

hidden_size = 128
num_layers = 2

# 1. Model Vanilla RNN
model_rnn_compare = nn.Sequential(
    ReshapeForRNN(),
    nn.RNN(input_size=28, hidden_size=hidden_size, num_layers=num_layers,
           batch_first=True, dropout=0.3),
    ExtractRNNOutput(),
    SelectLastTimeStep(),
    nn.Linear(hidden_size, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 10)
).to(device)

# 2. Model LSTM
model_lstm_compare = nn.Sequential(
    ReshapeForRNN(),
    nn.LSTM(input_size=28, hidden_size=hidden_size, num_layers=num_layers,
            batch_first=True, dropout=0.3),
    ExtractLSTMOutput(),
    SelectLastTimeStep(),
    nn.Linear(hidden_size, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 10)
).to(device)

# 3. Model GRU
model_gru_compare = nn.Sequential(
    ReshapeForRNN(),
    nn.GRU(input_size=28, hidden_size=hidden_size, num_layers=num_layers,
           batch_first=True, dropout=0.3),
    ExtractRNNOutput(),
    SelectLastTimeStep(),
    nn.Linear(hidden_size, 64),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(64, 10)
).to(device)

# Hitung parameter
models = {
    'RNN': model_rnn_compare,
    'LSTM': model_lstm_compare,
    'GRU': model_gru_compare
}

print("=" * 50)
print(f"{'Model':<8} | {'Parameter':>15}")
print("-" * 50)
for name, model in models.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f"{name:<8} | {n_params:>15,}")
print("=" * 50)

In [ ]:
# ============================
# TRAINING LOOP — KETIGA MODEL
# ============================

num_epochs = 10
results = {}

print("\n" + "=" * 90)
print("TRAINING PERBANDINGAN: RNN vs LSTM vs GRU")
print("=" * 90)

for model_name, model in models.items():
    print(f"\n--- Training {model_name} ---")

    # Reset optimizer untuk setiap model
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    test_accs = []
    start_time = time.time()

    for epoch in range(num_epochs):
        # ---- TRAINING ----
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

        train_loss = running_loss / len(train_loader)
        train_losses.append(train_loss)

        # ---- TESTING ----
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_acc = correct / total * 100
        test_accs.append(test_acc)

        if (epoch + 1) % 2 == 0 or epoch == 0:
            print(f"  Epoch [{epoch+1:2d}/{num_epochs}] | "
                  f"Loss: {train_loss:.4f} | Acc: {test_acc:.2f}%")

    training_time = time.time() - start_time

    results[model_name] = {
        'train_losses': train_losses,
        'test_accs': test_accs,
        'final_acc': test_accs[-1],
        'time': training_time,
        'params': sum(p.numel() for p in model.parameters())
    }

    print(f"Selesai! Akurasi: {test_accs[-1]:.2f}% | "
          f"Waktu: {training_time:.1f}s")

print("\n" + "=" * 90)

In [ ]:
# ============================
# PLOT PERBANDINGAN
# ============================

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

colors = {'RNN': 'coral', 'LSTM': 'steelblue', 'GRU': 'green'}

# Plot 1: Training Loss
for name, res in results.items():
    axes[0].plot(range(1, num_epochs+1), res['train_losses'], 'o-',
                 color=colors[name], linewidth=2, label=name, markersize=5)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# Plot 2: Test Accuracy
for name, res in results.items():
    axes[1].plot(range(1, num_epochs+1), res['test_accs'], 's-',
                 color=colors[name], linewidth=2, label=name, markersize=5)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Test Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

# Plot 3: Perbandingan Final (Bar chart)
model_names = list(results.keys())
final_accs = [results[n]['final_acc'] for n in model_names]
bar_colors = [colors[n] for n in model_names]

bars = axes[2].bar(model_names, final_accs, color=bar_colors,
                   edgecolor='black', linewidth=0.5)
for bar, acc in zip(bars, final_accs):
    axes[2].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
                 f'{acc:.2f}%', ha='center', va='bottom',
                 fontsize=12, fontweight='bold')
axes[2].set_ylabel('Accuracy (%)', fontsize=12)
axes[2].set_title('Final Test Accuracy', fontsize=14, fontweight='bold')
axes[2].set_ylim(90, 100)
axes[2].grid(True, alpha=0.3, axis='y')

plt.suptitle('Perbandingan RNN vs LSTM vs GRU — Klasifikasi MNIST',
             fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ============================
# TABEL RINGKASAN
# ============================

print("\n" + "=" * 75)
print(f"{'Model':<8} | {'Parameter':>12} | {'Akurasi (%)':>12} | "
      f"{'Waktu (s)':>10} | {'Acc/Param':>12}")
print("-" * 75)

for name, res in results.items():
    acc_per_param = res['final_acc'] / res['params'] * 10000
    print(f"{name:<8} | {res['params']:>12,} | {res['final_acc']:>12.2f} | "
          f"{res['time']:>10.1f} | {acc_per_param:>12.4f}")

print("=" * 75)
print("\nInsight:")
print("   • LSTM dan GRU umumnya unggul dari RNN dasar berkat mekanisme gate")
print("   • GRU memiliki parameter lebih sedikit dari LSTM dan training lebih cepat")
print("   • Untuk MNIST (sekuens pendek), perbedaannya mungkin tidak signifikan")
print("   • Perbedaan lebih terlihat pada sekuens yang PANJANG (100+ time steps)")

In [ ]:
# ============================
# MODUL UNTUK BIDIRECTIONAL
# ============================

# Untuk Bidirectional, output size = 2 * hidden_size
# Hidden state shape juga berubah: (num_layers * 2, batch, hidden_size)

# SelectLastBidirectional: mengambil output terakhir dari bidirectional RNN
# Forward: output[:, -1, :hidden] + Backward: output[:, 0, hidden:]
# Atau lebih sederhana: ambil seluruh output di time step terakhir

class SelectLastBidirectional(nn.Module):
    """Mengambil output pada time step terakhir dari Bidirectional RNN.
    Input:  (batch, seq_len, 2*hidden_size)
    Output: (batch, 2*hidden_size)
    """
    __init__ = lambda self: super().__init__()
    forward = lambda self, x: x[:, -1, :]  # Ambil time step terakhir

print("SelectLastBidirectional berhasil didefinisikan!")

In [ ]:
# ============================
# MODEL: BIDIRECTIONAL LSTM UNTUK MNIST
# ============================

# Bidirectional LSTM membaca gambar dari ATAS ke BAWAH dan BAWAH ke ATAS
# sehingga memiliki konteks global dari seluruh baris gambar

model_bilstm = nn.Sequential(
    # Reshape
    ReshapeForRNN(),                     # (B,1,28,28) → (B,28,28)

    # Bidirectional LSTM
    nn.LSTM(input_size=28, hidden_size=128, num_layers=2,
            batch_first=True, dropout=0.3,
            bidirectional=True),          # ← BIDIRECTIONAL!
    ExtractLSTMOutput(),                  # (B, 28, 256) — 256 = 128*2
    SelectLastBidirectional(),            # (B, 256)

    # Classifier
    nn.Linear(256, 128),                  # 256 karena bidirectional (128*2)
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10)                    # 10 kelas
).to(device)

print("Arsitektur Model Bidirectional LSTM:")
print(model_bilstm)

total_params = sum(p.numel() for p in model_bilstm.parameters())
print(f"\nTotal parameter: {total_params:,}")

In [ ]:
# ============================
# TRAINING BIDIRECTIONAL LSTM
# ============================

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_bilstm.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

num_epochs = 10
train_losses = []
test_accs = []

print("Mulai Training Bidirectional LSTM...")
print("=" * 80)

for epoch in range(num_epochs):
    # ---- TRAINING ----
    model_bilstm.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model_bilstm(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model_bilstm.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = correct / total * 100
    train_losses.append(train_loss)

    # ---- TESTING ----
    model_bilstm.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_bilstm(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    test_acc = correct / total * 100
    test_accs.append(test_acc)
    scheduler.step()

    print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Test Acc: {test_acc:.2f}%")

print("=" * 80)
print(f"Akurasi Akhir Bidirectional LSTM: {test_accs[-1]:.2f}%")

In [ ]:
# ============================
# EVALUASI BIDIRECTIONAL LSTM
# ============================

model_bilstm.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_bilstm(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Classification Report
print("\nClassification Report — Bidirectional LSTM MNIST:")
print("=" * 60)
print(classification_report(all_labels, all_preds,
                             target_names=[f'Digit {i}' for i in range(10)]))

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[str(i) for i in range(10)],
            yticklabels=[str(i) for i in range(10)])
plt.xlabel('Prediksi', fontsize=13)
plt.ylabel('Aktual', fontsize=13)
plt.title('Confusion Matrix — Bidirectional LSTM MNIST',
          fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()